In [ ]:
RUN_FULL = False
import os, sys, json, time, subprocess, shutil, platform
import pandas as pd, torch
print("Python", platform.python_version(), "| torch", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu (no GPU)")
WORK = "/kaggle/working"
RESULTS = f"{WORK}/results_official"
os.makedirs(RESULTS, exist_ok=True)

In [ ]:
os.chdir(WORK)
if not os.path.isdir("CRRS"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/RUCAIBox/CRRS.git", "CRRS"], check=True)
p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "recbole==1.1.1", "torch_geometric"], capture_output=True, text=True)
print("pip exit code:", p.returncode)
print((p.stdout + p.stderr)[-1500:])
v = subprocess.run([sys.executable, "-c", "import recbole; print(recbole.__version__)"], capture_output=True, text=True)
print("RecBole version:", v.stdout.strip() or v.stderr[-500:])

In [ ]:
%%writefile run_official.py
# ---------------------------------------------------------------------------
# Runs one model with the OFFICIAL CRRS / RecBole-PJF code and saves its test metrics.
# Usage (from inside the cloned CRRS folder):
#   python run_official.py --model DPGNN --epochs 300 --out ../results/official_DPGNN.json
# The patches below only restore names removed by newer NumPy / pandas / PyTorch releases;
# they do not change any model, data or metric logic.
# ---------------------------------------------------------------------------
import argparse
import json
import os
import sys
import time

import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# CRRS is built on RecBole 1.1.1, which uses np.float, np.bool, np.long and np.str (removed in NumPy 2);
# RecBole 1.2.0 additionally reads np.float_, np.complex_ and np.unicode_. All are restored here.
for alias, real in {"float_": np.float64, "complex_": np.complex128, "unicode_": np.str_, "string_": np.bytes_,
                    "NaN": np.nan, "Inf": np.inf, "float": float, "int": int, "bool": bool,
                    "object": object, "complex": complex, "long": int, "str": str}.items():
    try:
        getattr(np, alias)
    except AttributeError:
        setattr(np, alias, real)

import pandas as pd
if not hasattr(pd.DataFrame, "append"):
    pd.DataFrame.append = lambda self, other, ignore_index=False, **kw: pd.concat(
        [self, other if isinstance(other, pd.DataFrame) else pd.DataFrame([other])], ignore_index=ignore_index)

import scipy.sparse as sp
if not hasattr(sp.dok_matrix, "_update"):      # RecBole 1.1.1 LightGCN calls dok_matrix._update (removed in SciPy 1.13+)
    def _dok_update(self, d):
        if hasattr(self, "_dict"):
            self._dict.update(d)
        else:
            dict.update(self, d)
    sp.dok_matrix._update = _dok_update

import torch
_orig_load = torch.load
def _load(*args, **kwargs):          # RecBole checkpoints store config objects, not just weights
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _load

sys.path.insert(0, os.getcwd())
from recbole_pjf.quick_start import run_recbole_pjf


def to_float(v):
    try:
        return float(v.item()) if hasattr(v, "item") else float(v)
    except Exception:
        return str(v)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="DPGNN")
    ap.add_argument("--epochs", type=int, default=300)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    t0 = time.time()
    res = run_recbole_pjf(model=args.model, dataset="libimseti",
                          config_dict={"epochs": args.epochs, "gpu_id": "0", "show_progress": False})
    side_a, side_b = res["test_result"]
    out = {"model": args.model, "epochs_cap": args.epochs, "seconds": time.time() - t0,
           "best_valid_score": to_float(res["best_valid_score"]),
           "side_A": {k: to_float(v) for k, v in side_a.items()},
           "side_B": {k: to_float(v) for k, v in side_b.items()}}
    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    with open(args.out, "w") as fh:
        json.dump(out, fh, indent=1)
    print(json.dumps(out, indent=1))


In [ ]:
shutil.copy("run_official.py", f"{WORK}/CRRS/run_official.py")

def run_official(model, epochs, out):
    log = out.replace(".json", ".log")
    t0 = time.time()
    with open(log, "w") as fh:
        p = subprocess.Popen([sys.executable, "run_official.py", "--model", model, "--epochs", str(epochs), "--out", out],
                             cwd=f"{WORK}/CRRS", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env={**os.environ, "PYTHONWARNINGS": "ignore"})
        for line in p.stdout:
            fh.write(line)
            if any(w in line for w in ("epoch", "best valid", "test result", "Error", "error", "Traceback")):
                print(line.rstrip()[:200])
        p.wait()
    ok = p.returncode == 0 and os.path.exists(out)
    print(f"official {model}: {'OK' if ok else 'FAILED (exit code %d)' % p.returncode} in {(time.time() - t0) / 60:.1f} min; log: {log}")
    if not ok:
        print("".join(open(log).readlines()[-30:]))
    return ok

In [ ]:
test_out = f"{RESULTS}/test_DPGNN_1epoch.json"
if os.path.exists(test_out):
    os.remove(test_out)
if run_official("DPGNN", 1, test_out):
    print(json.dumps(json.load(open(test_out)), indent=1))

In [ ]:
MODELS = ["DPGNN", "LFRR", "LightGCN", "BPR"]
EPOCHS = 300
if not RUN_FULL:
    print("RUN_FULL is False: skipping full training (set RUN_FULL = True in the first cell for the background run)")
else:
    for model in MODELS:
        out = f"{RESULTS}/official_{model}.json"
        if os.path.exists(out):
            print(f"skip {model} (already done)")
            continue
        run_official(model, EPOCHS, out)

In [ ]:
PUBLISHED = {"DPGNN": (0.1825, 0.0739, 0.3007), "LFRR": (0.1729, 0.0706, 0.3045),
             "LightGCN": (0.1510, 0.0634, 0.2549), "BPR": (0.1552, 0.0661, 0.2795)}
OURS = {"DPGNN": ("0.1808 ± 0.0049", "0.0732 ± 0.0015", "0.2965 ± 0.0041"),
        "LightGCN": ("0.1463 ± 0.0013", "0.0615 ± 0.0005", "0.2498 ± 0.0013"),
        "BPR": ("0.1508 ± 0.0022", "0.0619 ± 0.0003", "0.2767 ± 0.0025")}
rows = []
for model, pub in PUBLISHED.items():
    f = f"{RESULTS}/official_{model}.json"
    off = ("–", "–", "–")
    if os.path.exists(f):
        r = json.load(open(f)); a, b = r["side_A"], r["side_B"]
        off = (f"{(a['recall@50'] + b['recall@50']) / 2:.4f}", f"{(a['ndcg@50'] + b['ndcg@50']) / 2:.4f}", f"{a['crecall']:.4f}")
    ours = OURS.get(model, ("–", "–", "–"))
    for i, m in enumerate(["Recall@50", "NDCG@50", "CRecall@50"]):
        rows.append({"Model": model, "Metric": m, "Published": f"{pub[i]:.4f}", "Official code (our run)": off[i], "Our re-implementation (3 seeds)": ours[i]})
table = pd.DataFrame(rows).set_index(["Model", "Metric"])
table.to_csv(f"{RESULTS}/table_official_vs_ours.csv")
display(table)
shutil.make_archive(f"{WORK}/official_results", "zip", RESULTS)
print("Saved", f"{WORK}/official_results.zip")